# 05_1d_inversion

1D layered complex-gain inversion against the validated analytic magnetic
(Hx) line-source solver (rockem-suite's `magnetic_line_source_fields_layered`)
- no empymod, no phase/amplitude correction constants.

This notebook:
- loads run data (`Hx_data.rss`, `Hz_data.rss`) and extracts complex
  steady-state channel gains (trace/wavelet phasor ratio) for Hx and Hz
  across all selected frequencies,
- calibrates each Tx against the analytic solver evaluated on the exact
  blocky 1D model read from that Tx's own `sg.rss` column, setting a
  per-frequency data sigma floored at the solver's own validated
  FDTD-vs-analytic accuracy band (see rockem-suite's
  `doc/examples/validate_layered_1d_model` README),
- inverts a 1D layered isotropic model independently for each Tx with a
  Tikhonov penalty (discourages fitting past the calibrated noise floor),
  using differential evolution, dual annealing, or a deterministic
  Levenberg-Marquardt block inversion (`empy_blockinv`),
- interpolates the Tx-wise 1D models into a pseudo-2D resistivity section
  over a homogeneous background,
- exports results (including calibration and chi2) to NPZ/JSON and
  optional RSS/SEG-Y.

**IMPORTANT - receiver depth requirement**: the analytic solver used here
is confirmed WRONG (not just unstable - converges to a stable but
incorrect answer) when the receiver depth is within roughly 0.15-0.2 skin
depths of the source depth (see rockem-suite's gotchas doc). This
notebook's calibration/inversion step will raise a clear error for any Tx
whose receivers share the source's exact depth (`gz0 == sz0` in
`01_fw_setup`, this workshop's literal default). If you hit that error,
go back to `01_fw_setup` and give the receiver depth (`gz0`) a real offset
from the source depth (`sz0`) - a few tens of meters is comfortably safe
at typical geophysical EM frequencies/resistivities; the 2D forward
modelling and 2D FWI stages (01-04) are unaffected by this constraint.


In [ ]:
from pathlib import Path
import json
import os
import re
import signal
import sys
import traceback

import numpy as np
from IPython.display import display

try:
    import ipywidgets as ipw
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    from scipy.optimize import differential_evolution, dual_annealing
    from joblib import Parallel, delayed
except Exception as exc:
    raise RuntimeError(
        "Missing dependencies. Install with: pip install scipy ipywidgets plotly joblib"
    ) from exc

# Project root
ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.modules.workshop_config import load_config
CONFIG = load_config()
WORKSPACE = CONFIG.workspace
INV_2D_RUNS_DIR = CONFIG.inv_2d_runs_dir

from scripts.modules import rockem_bridge
from scripts.modules.analytic_1d_forward import (
    ForwardRejected,
    Layer1D,
    check_kx_convergence,
    forward_1d_gains,
)
from scripts.modules.fd_visualization import (
    compute_gains_for_fd_outputs,
    load_rss_traces,
)
from scripts.modules.segy import write_resistivity_to_segy_from_template
from third_party.empy_blockinv import (
    InversionConfig,
    pack_model,
    run_block1d_inversion,
    split_model,
    uncertainty_from_result,
)
from third_party.rockseis.io.rsfile import rsfile

RUN_DIR_PATTERN = re.compile(r"^Run(\d+)$")
RUN_1D_DIR_PATTERN = re.compile(r"^Run(\d+)$")
INV_2D_INPUT_DIR = CONFIG.inv_2d_input_dir
FWD_2D_DIR = CONFIG.fwd_2d_dir
FDMODEL_DATA_DIR = FWD_2D_DIR / "Data"
SG_TRUE_PATH = FWD_2D_DIR / "sg.rss"
SETUP_META = FWD_2D_DIR / "setup_metadata.json"
INV_1D_RUNS_DIR = CONFIG.inv_1d_runs_dir
VOILA_PID_FILE = ROOT / ".voila_1d_inversion_server.pid"

state = {
    "features": None,
    "tx_results": {},
    "section": None,
    "messages": [],
    "active_run_dir": None,
    "forward_data_dim": 2,
    "calibration": None,
}


def push_message(msg):
    state["messages"].append(str(msg))
    if len(state["messages"]) > 20:
        state["messages"] = state["messages"][-20:]
    status_out.value = "\n".join(state["messages"])


def list_run_dirs(root_dir):
    out = []
    for child in Path(root_dir).iterdir():
        if child.is_dir():
            m = RUN_DIR_PATTERN.match(child.name)
            if m:
                out.append((int(m.group(1)), child))
    out.sort(key=lambda x: x[0])
    return out


def list_1d_run_dirs(base_dir=None):
    base = Path(base_dir) if base_dir else INV_1D_RUNS_DIR
    if not base.exists():
        return []
    out = []
    for child in base.iterdir():
        if child.is_dir():
            m = RUN_1D_DIR_PATTERN.match(child.name)
            if m:
                out.append((int(m.group(1)), child))
    out.sort(key=lambda x: x[0])
    return out


def find_next_1d_run_dir(base_dir=None):
    used = {idx for idx, _ in list_1d_run_dirs(base_dir)}
    idx = 0
    while idx in used:
        idx += 1
    base = Path(base_dir) if base_dir else INV_1D_RUNS_DIR
    return base / f"Run{idx}"


def write_1d_run_metadata(run_dir, cfg, n_runs, seeds, extra=None):
    import datetime
    meta = {
        "run_dir": str(run_dir),
        "timestamp": datetime.datetime.now().isoformat(),
        "n_runs_per_tx": int(n_runs),
        "seed_base": int(cfg["seed"]),
        "seeds": [int(s) for s in seeds],
        "config": {k: (v.tolist() if hasattr(v, "tolist") else v) for k, v in cfg.items()},
    }
    if extra:
        meta.update(extra)
    p = Path(run_dir) / "run_metadata.json"
    p.write_text(json.dumps(meta, indent=2))
    return p


def read_setup_metadata():
    if not SETUP_META.exists():
        return {}
    try:
        return json.loads(SETUP_META.read_text())
    except Exception:
        return {}


def get_forward_data_dim():
    meta = read_setup_metadata()
    try:
        return int(meta.get("forward_data_dim", 2))
    except Exception:
        return 2


def get_eps_r_used():
    """The forward run's own `eps_r_used` (from `design_explicit_fd` /
    `01_fw_setup`'s setup_metadata.json) - the analytic solver MUST use the
    same eps_r as the FDTD "truth" data, or the residual would conflate
    genuine discretization error with an eps_r mismatch on top of it (see
    `analytic_1d_forward.forward_1d_gains`'s docstring)."""
    meta = read_setup_metadata()
    v = meta.get("eps_r_used")
    return float(v) if v else 7.0


def get_f_min_hz(freqs):
    meta = read_setup_metadata()
    v = meta.get("f_min_hz")
    if v:
        return float(v)
    return float(np.min(freqs)) if len(freqs) else 2000.0


def default_frequencies():
    setup_meta = ROOT / "workspace/2D/forward" / "setup_metadata.json"
    if setup_meta.exists():
        try:
            meta = json.loads(setup_meta.read_text())
            vals = [float(v) for v in meta.get("flist_hz", [])]
            if vals:
                return vals
        except Exception:
            pass
    return [2e3, 4e3, 6e3]


def parse_freqs(text):
    vals = [float(v.strip()) for v in str(text).split(",") if v.strip()]
    if not vals:
        raise ValueError("Frequency list cannot be empty.")
    arr = np.asarray(vals, dtype=float)
    if np.any(arr <= 0):
        raise ValueError("Frequencies must be positive.")
    return arr


def wrap_phase_rad(phi):
    return np.arctan2(np.sin(phi), np.cos(phi))


def conductivity_to_resistivity(grid, min_sigma=1e-12):
    sigma = np.clip(np.asarray(grid, dtype=float), min_sigma, 1e12)
    return 1.0 / sigma


def _read_rss_model(path):
    f = rsfile()
    f.read(str(path))
    data = np.asarray(f.data, dtype=float)

    # RSS models can be x-z (2D) or x-y-z (3D). For 2.5D workflows,
    # ignore y and use a representative x-z slice.
    if data.ndim == 3:
        iy = int(data.shape[1] // 2)
        data_xz = np.asarray(data[:, iy, :], dtype=float)
    else:
        data_xz = np.squeeze(data)
        if data_xz.ndim == 3:
            iy = int(data_xz.shape[1] // 2)
            data_xz = np.asarray(data_xz[:, iy, :], dtype=float)

    if data_xz.ndim != 2:
        raise ValueError(f"Expected 2D or 3D model, got shape {data.shape}")

    nx, nz = int(data_xz.shape[0]), int(data_xz.shape[1])
    grid = np.asarray(data_xz.T, dtype=float)
    dx = float(f.geomD[0]) if f.geomD[0] else 1.0
    ox = float(f.geomO[0])
    iz = 2 if (len(f.geomN) > 2 and int(f.geomN[2]) > 0) else 1
    dz = float(f.geomD[iz]) if f.geomD[iz] else 1.0
    oz = float(f.geomO[iz])
    x = ox + dx * np.arange(nx)
    z = oz + dz * np.arange(nz)
    return x, z, grid


# -------------------- True-model forward (sg.rss) --------------------
TRUE_RSS_Z_POSITIVE_UP = False


def _blocky_layers_from_trace(rho_cells, z0, dz, rtol=1e-5, atol=0.0):
    """Merge adjacent equal rho cells into empymod (depth,res).

    Returns
    - depth_abs: (nlay-1,) interface depths (absolute, positive down)
    - res: (nlay,) layer resistivities
    """
    rho = np.asarray(rho_cells, dtype=float).ravel()
    nz = rho.size
    if nz == 0:
        return np.array([], dtype=float), np.array([], dtype=float)

    cell_bottom = float(z0) + (np.arange(nz, dtype=float) + 1.0) * float(dz)
    is_new = np.ones(nz, dtype=bool)
    is_new[1:] = ~np.isclose(rho[1:], rho[:-1], rtol=rtol, atol=atol)
    run_starts = np.flatnonzero(is_new)

    res = np.empty(run_starts.size, dtype=float)
    depth = np.empty(max(0, run_starts.size - 1), dtype=float)
    for j, s in enumerate(run_starts):
        e = nz - 1 if j == run_starts.size - 1 else int(run_starts[j + 1]) - 1
        res[j] = rho[int(s)]
        if j < run_starts.size - 1:
            depth[j] = cell_bottom[int(e)]
    return depth, res


_true_model_cache = {
    # keyed by (tx_x_rounded, tx_z_rounded): dict(depth_rel,res)
}


def true_model_layers_for_tx(tx_entry):
    """Return (depth_rel,res) for the true model beneath this Tx."""
    key = (round(float(tx_entry.get("tx_x", 0.0)), 6), round(float(tx_entry.get("tx_z", 0.0)), 6))
    if key in _true_model_cache:
        rec = _true_model_cache[key]
        return rec["depth_rel"], rec["res"]

    # Read sg.rss conductivity grid; convert to resistivity.
    xg, zg, sigma_grid = _read_rss_model(SG_TRUE_PATH)
    rho_grid = conductivity_to_resistivity(sigma_grid)

    # Pick nearest x column to transmitter x.
    tx_x = float(tx_entry.get("tx_x", 0.0))
    ix = int(np.argmin(np.abs(np.asarray(xg, dtype=float) - tx_x))) if np.size(xg) else 0

    rho_trace = np.asarray(rho_grid[:, ix], dtype=float)

    # Interpret z grid as cell-top origins with constant dz.
    z0 = float(zg[0]) if np.size(zg) else 0.0
    dz = float(zg[1] - zg[0]) if np.size(zg) > 1 else 1.0
    if TRUE_RSS_Z_POSITIVE_UP:
        z0 = -z0
        dz = -dz
    if dz <= 0.0:
        raise ValueError("True-model vertical dz must be > 0 after mapping.")

    depth_abs, res = _blocky_layers_from_trace(rho_trace, z0=z0, dz=dz)

    # Shift absolute interfaces to Tx-centered depth (match single_and_crossplot.py).
    tx_z = float(tx_entry.get("tx_z", 0.0))
    depth_rel = np.asarray(depth_abs, dtype=float) - tx_z

    _true_model_cache[key] = {"depth_rel": depth_rel, "res": res}
    return depth_rel, res


def _extract_positions():
    candidates = [
        FDMODEL_DATA_DIR / "Hxshot.rss",
        FDMODEL_DATA_DIR / "Hx_data.rss",
        INV_2D_INPUT_DIR / "Hx_data.rss",
    ]

    hx_path = None
    for c in candidates:
        if c.exists():
            hx_path = c
            break
    if hx_path is None:
        return np.array([]), np.array([]), np.array([]), np.array([])

    try:
        meta = load_rss_traces(hx_path)
        src = np.column_stack((meta["src_x"], meta["src_z"]))
        rec = np.column_stack((meta["rx_x"], meta["rx_z"]))
        src_u = np.unique(np.round(src, 6), axis=0)
        rec_u = np.unique(np.round(rec, 6), axis=0)
        return src_u[:, 0], src_u[:, 1], rec_u[:, 0], rec_u[:, 1]
    except Exception:
        return np.array([]), np.array([]), np.array([]), np.array([])


def get_real_data_paths():
    candidates = [
        (FDMODEL_DATA_DIR / "Hxshot.rss", FDMODEL_DATA_DIR / "Hzshot.rss"),
        (FDMODEL_DATA_DIR / "Hx_data.rss", FDMODEL_DATA_DIR / "Hz_data.rss"),
        (INV_2D_INPUT_DIR / "Hx_data.rss", INV_2D_INPUT_DIR / "Hz_data.rss"),
    ]

    for hx, hz in candidates:
        if hx.exists() and hz.exists():
            return hx, hz

    # Return first pair for clearer error paths upstream.
    return candidates[0]


def extract_features(freqs, f_min_hz=None, n_periods_extract=3.0):
    """Single source of truth for FD feature extraction: complex channel
    gains (trace/wavelet phasor ratio, see `fd_visualization.steady_state_
    gains`) per Tx, no phase-correction constants, no amplitude-spreading
    heuristic - the analytic Hx-source solver
    (`analytic_1d_forward.forward_1d_gains`) gives the exact 2D-vs-3D
    reconciliation directly, so none of that is needed here anymore.
    """
    hx_path, hz_path = get_real_data_paths()
    if not hx_path.exists() or not hz_path.exists():
        raise FileNotFoundError("Hx/Hz observation data not found in workspace/2D/forward/Data or workspace/2D/inversion/input")

    wav_path = FWD_2D_DIR / "wav2d.rss"
    if not wav_path.exists():
        raise FileNotFoundError(f"Wavelet file not found: {wav_path}")

    freqs = np.asarray(freqs, dtype=float)
    f_min_hz = float(f_min_hz) if f_min_hz else get_f_min_hz(freqs)
    gains = compute_gains_for_fd_outputs(
        hx_path, hz_path, wav_path, freqs=freqs, f_min_hz=f_min_hz, n_periods_extract=n_periods_extract,
    )
    geo = gains["geometry"]
    tx_idx = np.asarray(geo["tx_idx_per_trace"], dtype=int)
    tx_unique = np.asarray(geo["tx_unique"], dtype=float)

    gain_hx = np.asarray(gains["Hx"]["gain"], dtype=complex)
    gain_hz = np.asarray(gains["Hz"]["gain"], dtype=complex)

    fd_input_data_dim = int(get_forward_data_dim())

    tx_data = {}
    for tx_id in np.unique(tx_idx):
        tr = np.where(tx_idx == int(tx_id))[0]
        if tr.size == 0:
            continue

        tx_xy = tx_unique[int(tx_id)]
        rx_x = np.asarray(geo["rx_x"], dtype=float)[tr]
        rx_z = np.asarray(geo["rx_z"], dtype=float)[tr]
        off_x = rx_x - float(tx_xy[0])
        off_z = rx_z - float(tx_xy[1])

        tx_data[int(tx_id)] = {
            "tx_id": int(tx_id),
            "tx_x": float(tx_xy[0]),
            "tx_z": float(tx_xy[1]),
            "rx_x": rx_x,
            "rx_z": rx_z,
            "off_x": off_x,
            "off_z": off_z,
            "freqs": np.asarray(freqs, dtype=float),
            "obs_hx_gain": np.asarray(gain_hx[:, tr], dtype=complex),
            "obs_hz_gain": np.asarray(gain_hz[:, tr], dtype=complex),
        }

    return {
        "run_dir": str(FDMODEL_DATA_DIR),
        "hx_path": str(hx_path),
        "hz_path": str(hz_path),
        "wav_path": str(wav_path),
        "freqs": np.asarray(freqs, dtype=float),
        "f_min_hz": float(f_min_hz),
        "tx_data": tx_data,
        "geometry": geo,
        "forward_data_dim": fd_input_data_dim,
    }


def unpack_model_params(params, n_layers, z_start_rel, z_end_rel):
    p = np.asarray(params, dtype=float)
    lrho = p[:n_layers]
    rho = np.power(10.0, lrho)

    # Thickness parameters are optimized in log10-space.
    lthk = p[n_layers:]
    thk_raw = np.power(10.0, lthk)
    thk_raw = np.clip(thk_raw, 1e-9, np.inf)

    span = float(z_end_rel - z_start_rel)
    if span <= 0:
        raise ValueError("Invalid depth window: z_end_rel must be > z_start_rel")

    if thk_raw.size != max(0, n_layers - 1):
        raise ValueError("Thickness parameter count mismatch.")

    if thk_raw.size > 0:
        thk = thk_raw / np.sum(thk_raw) * span
        depth = float(z_start_rel) + np.cumsum(thk)
    else:
        thk = np.array([], dtype=float)
        depth = np.array([], dtype=float)
    return rho, thk, depth


def forward_analytic_for_tx(params, tx_entry, n_layers, z_start_rel, z_end_rel, eps_r, n_nodes=120):
    """Analytic (Hx, Hz) complex channel gain for a candidate model, via
    `analytic_1d_forward.forward_1d_gains` - the validated 2D magnetic
    line-source solver, exact counterpart of the FDTD survey (see that
    module's docstring). Raises `ForwardRejected` on an unevaluable model -
    callers must catch it (never let it silently become NaN)."""
    rho, thk, _ = unpack_model_params(params, n_layers, z_start_rel, z_end_rel)
    tx_z = float(tx_entry["tx_z"])
    off_x = np.asarray(tx_entry["off_x"], dtype=float)
    off_z = np.asarray(tx_entry["off_z"], dtype=float)
    rx_depth_m = tx_z + float(off_z[0]) if off_z.size else tx_z
    return forward_1d_gains(rho, thk, tx_entry["freqs"], off_x, tx_z, rx_depth_m, eps_r, n_nodes=n_nodes)


def forward_analytic_true_model_for_tx(tx_entry, eps_r, n_nodes=120):
    """Analytic forward for the TRUE model beneath this Tx (from `sg.rss`),
    used only as a QC overlay - see `true_model_layers_for_tx`. QC-only
    reconstruction: `true_model_layers_for_tx` gives interior interfaces
    only, tx-relative (empymod's top/bottom-halfspace convention), but
    `Layer1D` needs an explicit finite thickness for layer 0 (every layer
    except the last), and `layers_to_empymod` auto-CENTERS the stack on
    `tx_depth_m` (`z0 = tx_depth_m - 0.5*span`). Solve for the layer-0
    thickness that makes that auto-centered stack land its first interior
    interface exactly at the known `depth_rel[0]` - i.e. invert the
    centering formula rather than guessing a thickness, so this reproduces
    the true absolute geometry exactly, not approximately.
    """
    depth_rel, res_true = true_model_layers_for_tx(tx_entry)
    res_true = np.asarray(res_true, dtype=float)
    depth_rel = np.asarray(depth_rel, dtype=float)
    interior_thk = np.diff(depth_rel) if depth_rel.size > 1 else np.array([], dtype=float)
    if res_true.size > 1:
        thk0 = 2.0 * float(depth_rel[0]) + float(np.sum(interior_thk))
        if thk0 <= 0.0:
            raise ForwardRejected("true-model layer-0 thickness solve gave a non-positive value "
                                   "(tx sits above the first interface - QC overlay not meaningful here)")
        thickness = np.concatenate([[thk0], interior_thk])
    else:
        thickness = np.array([], dtype=float)
    tx_z = float(tx_entry["tx_z"])
    off_x = np.asarray(tx_entry["off_x"], dtype=float)
    off_z = np.asarray(tx_entry["off_z"], dtype=float)
    rx_depth_m = tx_z + float(off_z[0]) if off_z.size else tx_z
    layers = [Layer1D(float(res_true[i]), float(thickness[i]) if i < thickness.size else None, float(eps_r))
              for i in range(res_true.size)]
    layers[-1].thickness_m = None
    _, hx, hz = magnetic_line_source_fields_layered_or_reject(off_x, tx_entry["freqs"], layers, tx_z, rx_depth_m, n_nodes)
    return hx, hz


def magnetic_line_source_fields_layered_or_reject(off_x, freqs_hz, layers, tx_depth_m, rx_depth_m, n_nodes):
    """Loop `rockem_bridge.magnetic_line_source_fields_layered` over
    frequency for an already-built `Layer1D` stack (used by the true-model
    QC overlay, which doesn't go through `forward_1d_gains`'s rho/thickness
    parameterization)."""
    freqs_hz = np.asarray(freqs_hz, dtype=float)
    nfreq, nrx = freqs_hz.size, np.asarray(off_x).size
    ey = np.full((nfreq, nrx), np.nan, dtype=complex)
    hx = np.full((nfreq, nrx), np.nan, dtype=complex)
    hz = np.full((nfreq, nrx), np.nan, dtype=complex)
    for i, f in enumerate(freqs_hz):
        try:
            e_f, hx_f, hz_f = rockem_bridge.magnetic_line_source_fields_layered(
                off_x, float(f), layers, float(tx_depth_m), rx_depth_m=float(rx_depth_m), n_nodes=n_nodes,
            )
        except rockem_bridge.GreensSolverError as exc:
            raise ForwardRejected(str(exc)) from exc
        ey[i, :], hx[i, :], hz[i, :] = e_f, hx_f, hz_f
    return ey, hx, hz


def complex_gain_objective(
    params, tx_entry, n_layers, z_start_rel, z_end_rel, eps_r,
    reg_lambda, w_hxh, w_hxhz, sigma_hx, sigma_hz, calibration_mode, C=None,
):
    """Complex-gain misfit: ((C*pred - obs)/sigma) summed in quadrature over
    frequency and receiver, for both Hx and Hz, plus a Tikhonov first-
    difference penalty on log10(rho). Replaces the old unit-circle
    phase-only objective and the amp+phase "reconciliation patch" - no
    phase-correction constants, no offset-power amplitude spreading, no
    forced time-derivative: `forward_analytic_for_tx` already gives the
    exact 2D line-source answer, and any residual calibration is a genuine,
    MEASURED FDTD-vs-analytic discretization effect (see `run_calibration`),
    not a guessed correction.

    A SINGLE complex constant `C` is shared by Hx and Hz (not one each) -
    checked directly across 4 different FD designs/resistivity regimes
    (near-offset/high-eps_r, conductive rho~2, resistive/low-freq, the
    repo's own canonical validation example): |C_hx|/|C_hz| never exceeded
    1.6% from 1.0, and forcing a single shared C only costs a few tenths of
    a percentage point of extra residual - negligible next to the 3%
    `VALIDATED_REL_ERROR_FLOOR` already applied to sigma. Per-component
    `sigma_hx`/`sigma_hz` stay separate (that's a genuine per-field noise
    floor, not the calibration constant).

    `calibration_mode`: "fixed" applies the measured per-frequency complex
    constant `C` to both predictions before differencing; "source_factor"
    instead solves for the best-fit complex scale per frequency in closed
    form, pooling Hx and Hz together the same way `run_calibration` does
    (`s_f = sum(conj(pred)*obs) / sum(|pred|^2)` summed over BOTH
    components) - use when the fixed calibration constant was found to be
    unstable (see `run_calibration`'s scatter report).
    """
    try:
        hx_pred, hz_pred = forward_analytic_for_tx(params, tx_entry, n_layers, z_start_rel, z_end_rel, eps_r)
    except ForwardRejected:
        return 1e12

    obs_hx = np.asarray(tx_entry["obs_hx_gain"], dtype=complex)
    obs_hz = np.asarray(tx_entry["obs_hz_gain"], dtype=complex)

    if calibration_mode == "source_factor":
        num = np.sum(np.conj(hx_pred) * obs_hx, axis=1, keepdims=True) + np.sum(np.conj(hz_pred) * obs_hz, axis=1, keepdims=True)
        den = np.sum(np.abs(hx_pred) ** 2, axis=1, keepdims=True) + np.sum(np.abs(hz_pred) ** 2, axis=1, keepdims=True)
        cal = num / np.where(den > 0, den, 1.0)
    else:
        cal = np.asarray(C, dtype=complex)[:, None] if C is not None else 1.0

    res_x = (cal * hx_pred - obs_hx) / np.maximum(np.asarray(sigma_hx, dtype=float)[:, None], 1e-300)
    res_z = (cal * hz_pred - obs_hz) / np.maximum(np.asarray(sigma_hz, dtype=float)[:, None], 1e-300)

    m1, m2 = np.isfinite(res_x), np.isfinite(res_z)
    if not np.any(m1) and not np.any(m2):
        return 1e12

    mis = 0.0
    if np.any(m1):
        mis += float(w_hxh) * float(np.nansum(np.where(m1, np.abs(res_x) ** 2, np.nan)))
    if np.any(m2):
        mis += float(w_hxhz) * float(np.nansum(np.where(m2, np.abs(res_z) ** 2, np.nan)))

    if reg_lambda > 0.0 and n_layers > 1:
        lrho = np.asarray(params[:n_layers], dtype=float)
        mis += float(reg_lambda) * float(np.mean(np.diff(lrho) ** 2))

    return mis


def build_bounds(n_layers, log10_rho_min, log10_rho_max, log10_thk_min, log10_thk_max):
    b = [(float(log10_rho_min), float(log10_rho_max)) for _ in range(int(n_layers))]
    for _ in range(int(n_layers) - 1):
        b.append((float(log10_thk_min), float(log10_thk_max)))
    return b


def _normalize_thickness_to_span(thickness, span_target, thk_min=1e-6):
    thk = np.asarray(thickness, dtype=float).reshape(-1)
    if thk.size == 0:
        return thk
    thk = np.clip(thk, float(thk_min), np.inf)
    total = float(np.sum(thk))
    if total <= 0.0:
        return np.full_like(thk, float(span_target) / float(thk.size))
    return thk * (float(span_target) / total)


def _params_from_rho_thk(rho, thickness, cfg):
    rho = np.asarray(rho, dtype=float)
    thk = np.asarray(thickness, dtype=float)
    span = float(cfg["z_end_rel"] - cfg["z_start_rel"])
    thk_norm = _normalize_thickness_to_span(thk, span_target=span, thk_min=cfg["thk_min"])
    lrho = np.log10(np.clip(rho, 10.0 ** cfg["log10_rho_min"], 10.0 ** cfg["log10_rho_max"]))
    lthk = np.log10(np.clip(thk_norm, 10.0 ** cfg["log10_thk_min"], 10.0 ** cfg["log10_thk_max"]))
    if lrho.size == 0:
        return lthk
    return np.concatenate([lrho, lthk]).astype(float)


def _pack_complex(c1, c2):
    c1 = np.asarray(c1, dtype=complex)
    c2 = np.asarray(c2, dtype=complex)
    return np.concatenate([
        np.ravel(np.real(c1)),
        np.ravel(np.imag(c1)),
        np.ravel(np.real(c2)),
        np.ravel(np.imag(c2)),
    ]).astype(float)


def _apply_calibration(pred_hx, obs_hx, pred_hz, obs_hz, calibration_mode, C):
    """Apply the fixed calibration constant or solve the per-frequency
    source factor in closed form (see `complex_gain_objective`'s
    docstring) - shared by the DE/dual_annealing objective, the blockinv
    forward/residual pair, and the QC plotting cell, so all three see the
    exact same (single, Hx/Hz-SHARED) calibration treatment - see
    `complex_gain_objective`'s docstring for why one constant, not two.
    """
    if calibration_mode == "source_factor":
        num = np.sum(np.conj(pred_hx) * obs_hx, axis=1, keepdims=True) + np.sum(np.conj(pred_hz) * obs_hz, axis=1, keepdims=True)
        den = np.sum(np.abs(pred_hx) ** 2, axis=1, keepdims=True) + np.sum(np.abs(pred_hz) ** 2, axis=1, keepdims=True)
        cal = num / np.where(den > 0, den, 1.0)
    else:
        cal = np.asarray(C, dtype=complex)[:, None] if C is not None else 1.0
    return cal * pred_hx, cal * pred_hz


def _blockinv_forward(model, ctx):
    n_layers = int(ctx["n_layers"])
    tx_entry = ctx["tx_entry"]
    cfg = ctx["cfg"]
    thk, rho = split_model(np.asarray(model, dtype=float), n_layers)
    params = _params_from_rho_thk(rho, thk, cfg)
    try:
        hx_pred, hz_pred = forward_analytic_for_tx(
            params, tx_entry, n_layers, cfg["z_start_rel"], cfg["z_end_rel"], cfg["eps_r"],
        )
    except ForwardRejected:
        # blockinv's Gauss-Newton step needs a finite (if bad) response, not
        # an exception - return a huge, finite, obviously-wrong prediction.
        nfreq, nrx = tx_entry["freqs"].size, tx_entry["off_x"].size
        hx_pred = np.full((nfreq, nrx), 1e6, dtype=complex)
        hz_pred = np.full((nfreq, nrx), 1e6, dtype=complex)

    cal = ctx["calibration"]
    obs_hx = np.asarray(tx_entry["obs_hx_gain"], dtype=complex)
    obs_hz = np.asarray(tx_entry["obs_hz_gain"], dtype=complex)
    hx_cal, hz_cal = _apply_calibration(hx_pred, obs_hx, hz_pred, obs_hz, cfg["calibration_mode"], cal.get("C"))
    pred_vec = _pack_complex(hx_cal, hz_cal)

    n_reg = int(ctx.get("n_reg", 0))
    reg_scale = float(ctx.get("reg_scale", 0.0))
    if n_reg > 0 and reg_scale > 0.0:
        lrho = np.asarray(params[:n_layers], dtype=float)
        reg_pred = reg_scale * np.diff(lrho)
        pred_vec = np.concatenate([pred_vec, np.asarray(reg_pred, dtype=float)])

    return pred_vec.astype(float)


def _blockinv_residual(observed, predicted, error, ctx):
    obs = np.asarray(observed, dtype=float)
    pred = np.asarray(predicted, dtype=float)
    err = np.asarray(error, dtype=float)
    res = (pred - obs) / np.clip(err, 1e-12, np.inf)

    n1 = int(ctx.get("n_hx", 0))
    n2 = int(ctx.get("n_hz", 0))
    w1 = np.sqrt(max(float(ctx.get("w_hxh", 1.0)), 0.0))
    w2 = np.sqrt(max(float(ctx.get("w_hxhz", 1.0)), 0.0))

    if n1 > 0 or n2 > 0:
        i0 = 0
        i1 = i0 + n1
        i2 = i1 + n1
        i3 = i2 + n2
        i4 = i3 + n2
        if i4 <= res.size:
            res[i0:i1] *= w1  # real(Hx)
            res[i1:i2] *= w1  # imag(Hx)
            res[i2:i3] *= w2  # real(Hz)
            res[i3:i4] *= w2  # imag(Hz)

    return res.astype(float)


def _invert_single_tx_blockinv(tx_entry, cfg):
    n_layers = int(cfg["n_layers"])
    cal = cfg["calibration"]
    obs_hx = np.asarray(tx_entry["obs_hx_gain"], dtype=complex)
    obs_hz = np.asarray(tx_entry["obs_hz_gain"], dtype=complex)
    obs_vec_data = _pack_complex(obs_hx, obs_hz)

    sigma_hx = np.broadcast_to(np.asarray(cal["sigma_hx"], dtype=float)[:, None], obs_hx.shape)
    sigma_hz = np.broadcast_to(np.asarray(cal["sigma_hz"], dtype=float)[:, None], obs_hz.shape)
    err_vec_data = np.concatenate([
        np.ravel(sigma_hx), np.ravel(sigma_hx), np.ravel(sigma_hz), np.ravel(sigma_hz),
    ]).astype(float)
    err_vec_data = np.clip(err_vec_data, 1e-300, np.inf)

    n_reg = max(n_layers - 1, 0)
    reg_scale = np.sqrt(max(float(cfg.get("reg_lambda", 0.0)), 0.0) / float(n_reg)) if n_reg > 0 else 0.0
    if n_reg > 0 and reg_scale > 0.0:
        obs_vec = np.concatenate([obs_vec_data, np.zeros(n_reg, dtype=float)])
        err_vec = np.concatenate([err_vec_data, np.ones(n_reg, dtype=float)])
    else:
        obs_vec = obs_vec_data
        err_vec = err_vec_data

    # Random initial model for blockinv (same across all Tx for a given seed).
    rng = np.random.default_rng(int(cfg["seed"]))

    rho_lo_l = float(cfg["log10_rho_min"])
    rho_hi_l = float(cfg["log10_rho_max"])
    start_rho = 10.0 ** rng.uniform(rho_lo_l, rho_hi_l, size=n_layers)

    thk_lo_l = float(cfg["log10_thk_min"])
    thk_hi_l = float(cfg["log10_thk_max"])
    start_thk = 10.0 ** rng.uniform(thk_lo_l, thk_hi_l, size=max(n_layers - 1, 0))

    span = float(cfg["z_end_rel"] - cfg["z_start_rel"])
    if start_thk.size > 0:
        start_thk = _normalize_thickness_to_span(
            start_thk, span_target=span, thk_min=cfg["thk_min"]
        )
    start_model = pack_model(start_thk, start_rho)

    callback_context = {
        "n_layers": n_layers,
        "tx_entry": tx_entry,
        "cfg": cfg,
        "calibration": cal,
        "n_hx": int(np.size(obs_hx)),
        "n_hz": int(np.size(obs_hz)),
        "w_hxh": float(cfg["w_hxh"]),
        "w_hxhz": float(cfg["w_hxhz"]),
        "n_reg": int(n_reg if reg_scale > 0.0 else 0),
        "reg_scale": float(reg_scale),
    }

    inv_cfg = InversionConfig(
        n_layers=n_layers,
        start_model=start_model,
        max_iter=int(cfg.get("block_max_iter", 15)),
        min_dphi_percent=1.0,
        stop_at_chi1=False,
        callback_context=callback_context,
        lower_bound=1e-8,
    )

    result = run_block1d_inversion(
        observed=obs_vec,
        error=err_vec,
        config=inv_cfg,
        forward_fn=_blockinv_forward,
        misfit_fn=_blockinv_residual,
    )

    thk_opt, rho_opt = split_model(np.asarray(result.model, dtype=float), n_layers)
    thk_opt = _normalize_thickness_to_span(thk_opt, span_target=span, thk_min=cfg["thk_min"])
    params_best = _params_from_rho_thk(rho_opt, thk_opt, cfg)
    try:
        pred_hxh, pred_hxhz = forward_analytic_for_tx(
            params_best, tx_entry, n_layers, cfg["z_start_rel"], cfg["z_end_rel"], cfg["eps_r"],
        )
    except ForwardRejected:
        pred_hxh = np.full_like(obs_hx, np.nan, dtype=complex)
        pred_hxhz = np.full_like(obs_hz, np.nan, dtype=complex)
    rho, thk, depth_rel = unpack_model_params(params_best, n_layers, cfg["z_start_rel"], cfg["z_end_rel"])

    sigma_rho = np.full_like(rho, np.nan, dtype=float)
    sigma_thk = np.full_like(thk, np.nan, dtype=float)
    sigma_depth = np.full_like(depth_rel, np.nan, dtype=float)
    try:
        sigma_abs, _ = uncertainty_from_result(result)
        sigma_abs = np.asarray(sigma_abs, dtype=float)
        n_thk = max(n_layers - 1, 0)
        sigma_thk_raw = sigma_abs[:n_thk] if n_thk else np.array([], dtype=float)
        sigma_rho_raw = sigma_abs[n_thk:]
        if n_thk:
            scale = float(span) / max(float(np.sum(np.asarray(result.model[:n_thk], dtype=float))), 1e-12)
            sigma_thk = np.asarray(sigma_thk_raw, dtype=float) * scale
            sigma_depth = np.sqrt(np.cumsum(np.maximum(sigma_thk, 0.0) ** 2))
        sigma_rho = np.asarray(sigma_rho_raw[: rho.size], dtype=float)
    except Exception:
        pass

    tx_z = float(tx_entry.get("tx_z", 0.0))
    z_top = tx_z + float(cfg["z_start_rel"])
    z_bottom = tx_z + float(cfg["z_end_rel"])
    depth_abs = tx_z + np.asarray(depth_rel, dtype=float)

    residual = np.asarray(result.residual, dtype=float)
    block_obj = float(np.dot(residual, residual)) if residual.size else np.nan
    chi2 = block_obj / max(residual.size, 1)

    return {
        "success": True,
        "message": "",
        "misfit": block_obj,
        "chi2": float(chi2),
        "params": params_best,
        "rho": rho,
        "thickness": thk,
        "depth": depth_abs,
        "z_top": float(z_top),
        "z_bottom": float(z_bottom),
        "pred_hxh": pred_hxh,
        "pred_hxhz": pred_hxhz,
        "rho_unc": sigma_rho,
        "thickness_unc": sigma_thk,
        "depth_unc": sigma_depth,
    }


def invert_single_tx(tx_entry, cfg):
    method = cfg["optimizer"]
    if method == "empy_blockinv":
        return _invert_single_tx_blockinv(tx_entry, cfg)

    bounds = build_bounds(
        cfg["n_layers"],
        cfg["log10_rho_min"],
        cfg["log10_rho_max"],
        cfg["log10_thk_min"],
        cfg["log10_thk_max"],
    )

    cal = cfg["calibration"]
    obj = lambda p: complex_gain_objective(
        p,
        tx_entry=tx_entry,
        n_layers=cfg["n_layers"],
        z_start_rel=cfg["z_start_rel"],
        z_end_rel=cfg["z_end_rel"],
        eps_r=cfg["eps_r"],
        reg_lambda=cfg["reg_lambda"],
        w_hxh=cfg["w_hxh"],
        w_hxhz=cfg["w_hxhz"],
        sigma_hx=cal["sigma_hx"],
        sigma_hz=cal["sigma_hz"],
        calibration_mode=cfg["calibration_mode"],
        C=cal.get("C"),
    )

    rng_seed = int(cfg["seed"])
    if method == "dual_annealing":
        out = dual_annealing(
            obj,
            bounds=bounds,
            maxfun=int(cfg["maxfun"]),
            seed=rng_seed,
        )
    else:
        out = differential_evolution(
            obj,
            bounds=bounds,
            maxiter=int(cfg["maxiter"]),
            popsize=int(cfg["popsize"]),
            seed=rng_seed,
            polish=False,
            workers=1,
            updating="deferred",
        )

    best = np.asarray(out.x, dtype=float)
    best_mis = float(out.fun)
    try:
        pred_hxh, pred_hxhz = forward_analytic_for_tx(
            best, tx_entry, cfg["n_layers"], cfg["z_start_rel"], cfg["z_end_rel"], cfg["eps_r"],
        )
    except ForwardRejected:
        nfreq, nrx = tx_entry["freqs"].size, tx_entry["off_x"].size
        pred_hxh = np.full((nfreq, nrx), np.nan, dtype=complex)
        pred_hxhz = np.full((nfreq, nrx), np.nan, dtype=complex)
    rho, thk, depth_rel = unpack_model_params(
        best,
        cfg["n_layers"],
        cfg["z_start_rel"],
        cfg["z_end_rel"],
    )

    n_data = 2 * pred_hxh.size + 2 * pred_hxhz.size
    chi2 = best_mis / max(n_data, 1)

    tx_z = float(tx_entry.get("tx_z", 0.0))
    z_top = tx_z + float(cfg["z_start_rel"])
    z_bottom = tx_z + float(cfg["z_end_rel"])
    depth_abs = tx_z + np.asarray(depth_rel, dtype=float)

    return {
        "success": bool(getattr(out, "success", True)),
        "message": str(getattr(out, "message", "")),
        "misfit": best_mis,
        "chi2": float(chi2),
        "params": best,
        "rho": rho,
        "thickness": thk,
        "depth": depth_abs,
        "z_top": float(z_top),
        "z_bottom": float(z_bottom),
        "pred_hxh": pred_hxh,
        "pred_hxhz": pred_hxhz,
    }


def build_z_grid_from_results(tx_results, dz=5.0, zmax=None):
    max_depth = 0.0
    for res in tx_results.values():
        if "depth" in res and np.asarray(res["depth"]).size:
            max_depth = max(max_depth, float(np.max(res["depth"])))
    if zmax is not None:
        max_depth = max(max_depth, float(zmax))
    max_depth = max(max_depth, float(dz))
    nz = int(np.ceil(max_depth / dz)) + 1
    return np.arange(nz, dtype=float) * float(dz)


def profile_from_result(res, z_grid, background_rho):
    rho = np.asarray(res.get("rho", []), dtype=float)
    interfaces = np.asarray(res.get("depth", []), dtype=float)
    z_top = float(res.get("z_top", np.min(z_grid) if z_grid.size else 0.0))
    z_bottom = float(res.get("z_bottom", np.max(z_grid) if z_grid.size else z_top))

    out = np.full_like(z_grid, float(background_rho), dtype=float)
    if rho.size == 0:
        return out

    z0 = z_top
    for i in range(rho.size):
        if i < interfaces.size:
            z1 = interfaces[i]
            m = (z_grid >= z0) & (z_grid < z1)
        else:
            z1 = z_bottom
            m = (z_grid >= z0) & (z_grid <= z1)
        out[m] = rho[i]
        z0 = z1
    return out


def interpolate_tx_profiles(tx_results, x_grid, z_grid, background_rho):
    if not tx_results:
        return np.full((z_grid.size, x_grid.size), float(background_rho), dtype=float)

    xs = []
    profiles = []
    ordered_ids = sorted(tx_results.keys())
    for tx_id in ordered_ids:
        res = tx_results[tx_id]
        xs.append(float(res["tx_x"]))
        profiles.append(profile_from_result(res, z_grid, background_rho=background_rho))

    xs = np.asarray(xs, dtype=float)
    profiles = np.asarray(profiles, dtype=float)
    order = np.argsort(xs)
    xs = xs[order]
    profiles = profiles[order, :]

    sec = np.full((z_grid.size, x_grid.size), float(background_rho), dtype=float)

    # Single-Tx case: fill an aperture around Tx instead of a single x sample.
    if xs.size == 1:
        tx_res = tx_results[ordered_ids[0]]
        rx_x = np.asarray(tx_res.get("rx_x", []), dtype=float)
        if rx_x.size > 0 and np.any(np.isfinite(rx_x)):
            x_left = float(np.nanmin(rx_x))
            x_right = float(np.nanmax(rx_x))
            m = (x_grid >= x_left) & (x_grid <= x_right)
        else:
            m = np.zeros_like(x_grid, dtype=bool)

        # If aperture is too narrow relative to grid, assign nearest x cell.
        if not np.any(m):
            ix = int(np.argmin(np.abs(x_grid - xs[0])))
            m[ix] = True

        sec[:, m] = profiles[0, :][:, None]
        return sec

    for iz in range(z_grid.size):
        sec[iz, :] = np.interp(
            x_grid,
            xs,
            profiles[:, iz],
            left=float(background_rho),
            right=float(background_rho),
        )
    return sec


def try_default_x_grid(run_dir):
    run_dir = Path(run_dir)
    candidates = [
        run_dir / "sg0.rss",
        run_dir / "sg_ls.rss",
        SG_TRUE_PATH,
    ]
    for p in candidates:
        if p.exists():
            try:
                x, _, _ = _read_rss_model(p)
                return x
            except Exception:
                pass
    return None


def get_default_model_axes():
    candidates = [
        SG_TRUE_PATH,
        FWD_2D_DIR / "sg0.rss",
        FWD_2D_DIR / "sg_ls.rss",
    ]
    for p in candidates:
        if p.exists():
            try:
                x, z, _ = _read_rss_model(p)
                return np.asarray(x, dtype=float), np.asarray(z, dtype=float)
            except Exception:
                pass
    return None, None


def default_depth_window():
    _, z = get_default_model_axes()
    if z is None or z.size == 0:
        return -200.0, 200.0
    return float(np.nanmin(z)), float(np.nanmax(z))


def write_rho_section_to_rss(template_path, out_path, rho_section):
    template_path = Path(template_path)
    out_path = Path(out_path)
    if not template_path.exists():
        raise FileNotFoundError(f"Template RSS not found: {template_path}")

    f = rsfile()
    f.read(str(template_path))
    data_t = np.asarray(f.data)

    if data_t.ndim == 3:
        nx_t, ny_t, nz_t = int(data_t.shape[0]), int(data_t.shape[1]), int(data_t.shape[2])
    else:
        data_2d = np.squeeze(data_t)
        if data_2d.ndim != 2:
            raise ValueError(f"Unsupported template RSS data shape: {data_t.shape}")
        nx_t, nz_t = int(data_2d.shape[0]), int(data_2d.shape[1])
        ny_t = 1

    rho = np.asarray(rho_section, dtype=float)
    if rho.shape != (nz_t, nx_t):
        raise ValueError(
            f"Section shape mismatch. Expected (nz={nz_t}, nx={nx_t}), got {rho.shape}"
        )

    sigma_xz = np.clip(1.0 / np.maximum(rho, 1e-9), 1e-12, 1e6).T
    if ny_t > 1:
        sigma_out = np.tile(sigma_xz[:, None, :], (1, ny_t, 1))
    else:
        sigma_out = sigma_xz

    f.data = np.asfortranarray(sigma_out.astype(data_t.dtype, copy=False))
    out_path.parent.mkdir(parents=True, exist_ok=True)
    f.write(str(out_path))


def _resolve_segy_template_path():
    if SETUP_META.exists():
        try:
            meta = json.loads(SETUP_META.read_text())
            p_raw = meta.get("segy_template_path")
            if p_raw:
                p = Path(p_raw).expanduser()
                if not p.is_absolute():
                    p = (ROOT / p).resolve()
                if p.exists():
                    return p, "setup_metadata.json"
                push_message(f"SEG-Y template listed in setup metadata does not exist: {p}")
        except Exception as exc:
            push_message(f"Failed reading setup metadata for SEG-Y template: {exc}")

    fallback = CONFIG.default_segy
    if fallback.exists():
        return fallback, "fallback default SEG-Y from config"
    return None, None


# -------------------- Calibration --------------------
# rockem-suite's own FDTD-vs-analytic validation of the Hx-source solver
# (doc/examples/validate_layered_1d_model/run_explicit_2d_hx_source.py)
# found ~0.5-2.1% amplitude error and a stable absolute channel-gain
# constant with <=2.1% scatter across offsets/components. That's the floor
# below which this workshop's own calibration (computed on whatever real
# model is loaded, often only 2 receivers - too few to estimate scatter
# reliably on its own) should never let sigma drop, so the inversion can
# never overfit past the solver's own known accuracy limit.
VALIDATED_REL_ERROR_FLOOR = 0.03


def run_calibration(tx_id, eps_r, reg_lambda_unused=None):
    """Compare this Tx's OBSERVED FDTD gain (already extracted from the
    real forward run) against the analytic solver evaluated on the exact
    blocky 1D model read from that same run's `sg.rss` beneath this Tx
    (`forward_analytic_true_model_for_tx`) - i.e. calibrate against the
    known model that ACTUALLY produced this data, no separate FDTD run
    needed. Returns a SINGLE per-frequency complex calibration constant
    `C`, shared by Hx and Hz (not one each - see `complex_gain_objective`'s
    docstring for the 4-scenario check justifying this), and per-frequency
    sigmas (`sigma_hx`, `sigma_hz`, kept separate - a genuine per-component
    noise floor, not the calibration constant) that are the larger of the
    measured scatter and the validated accuracy floor
    (`VALIDATED_REL_ERROR_FLOOR`) - see module docstring. If the real
    subsurface has genuine lateral variation this also (correctly) folds
    2D/3D model-approximation error into sigma, not just discretization
    error - a conservative, not an optimistic, sigma.
    """
    feats = state.get("features")
    if feats is None or tx_id not in feats["tx_data"]:
        raise RuntimeError("Load features first.")
    tx_entry = feats["tx_data"][tx_id]
    obs_hx = np.asarray(tx_entry["obs_hx_gain"], dtype=complex)
    obs_hz = np.asarray(tx_entry["obs_hz_gain"], dtype=complex)

    pred_hx, pred_hz = forward_analytic_true_model_for_tx(tx_entry, eps_r)

    num = np.sum(np.conj(pred_hx) * obs_hx, axis=1) + np.sum(np.conj(pred_hz) * obs_hz, axis=1)
    den = np.sum(np.abs(pred_hx) ** 2, axis=1) + np.sum(np.abs(pred_hz) ** 2, axis=1)
    C = num / np.where(den > 0, den, 1.0)

    def _sigma(obs, pred):
        resid = obs - C[:, None] * pred
        scatter = np.sqrt(np.mean(np.abs(resid) ** 2, axis=1))
        floor = VALIDATED_REL_ERROR_FLOOR * np.mean(np.abs(obs), axis=1)
        sigma = np.maximum(scatter, floor)
        rel_scatter = scatter / np.maximum(np.mean(np.abs(obs), axis=1), 1e-300)
        return sigma, rel_scatter

    sigma_hx, rel_hx = _sigma(obs_hx, pred_hx)
    sigma_hz, rel_hz = _sigma(obs_hz, pred_hz)

    return {
        "tx_id": int(tx_id), "eps_r": float(eps_r),
        "C": C, "sigma_hx": sigma_hx, "sigma_hz": sigma_hz,
        "rel_scatter_hx": rel_hx, "rel_scatter_hz": rel_hz,
        "notes": f"sigma floored at {VALIDATED_REL_ERROR_FLOOR * 100:.0f}% "
                 "(rockem-suite's own Hx-source FDTD-vs-analytic validation band)",
    }


# -------------------- UI --------------------
def_z_start, def_z_end = default_depth_window()

freqs_input = ipw.Text(
    value=",".join([f"{v:g}" for v in default_frequencies()]),
    description="Freqs (Hz):",
    layout=ipw.Layout(width="360px"),
)
n_periods_extract_input = ipw.FloatText(value=3.0, description="n_periods (of f_min)", layout=ipw.Layout(width="260px"))

n_layers_input = ipw.BoundedIntText(value=5, min=2, max=20, description="n_layers", layout=ipw.Layout(width="190px"))
z_start_input = ipw.FloatText(value=def_z_start, description="z_start", layout=ipw.Layout(width="210px"))
z_end_input = ipw.FloatText(value=def_z_end, description="z_end", layout=ipw.Layout(width="210px"))
background_rho_input = ipw.FloatText(value=10.0, description="rho_bg", layout=ipw.Layout(width="190px"))

rho_min_input = ipw.FloatText(value=1.0, description="rho_min", layout=ipw.Layout(width="180px"))
rho_max_input = ipw.FloatText(value=200.0, description="rho_max", layout=ipw.Layout(width="180px"))
thk_min_input = ipw.FloatText(value=5.0, description="thk_min", layout=ipw.Layout(width="180px"))
thk_max_input = ipw.FloatText(value=50.0, description="thk_max", layout=ipw.Layout(width="180px"))

optimizer_input = ipw.Dropdown(
    options=[
        ("Differential Evolution", "differential_evolution"),
        ("Dual Annealing", "dual_annealing"),
        ("BlockInv (pyGIMLi-style, deterministic)", "empy_blockinv"),
    ],
    value="differential_evolution",
    description="optimizer",
    layout=ipw.Layout(width="380px"),
)
calibration_mode_input = ipw.Dropdown(
    options=[("Fixed calibration constant", "fixed"), ("Per-frequency source factor", "source_factor")],
    value="fixed",
    description="calibration",
    layout=ipw.Layout(width="320px"),
)
maxiter_input = ipw.BoundedIntText(value=30, min=1, max=2000, description="maxiter", layout=ipw.Layout(width="170px"))
popsize_input = ipw.BoundedIntText(value=10, min=2, max=100, description="popsize", layout=ipw.Layout(width="170px"))
maxfun_input = ipw.BoundedIntText(value=800, min=20, max=100000, description="maxfun", layout=ipw.Layout(width="190px"))
seed_input = ipw.BoundedIntText(value=42, min=0, max=10_000_000, description="seed", layout=ipw.Layout(width="170px"))
n_runs_input = ipw.BoundedIntText(value=1, min=1, max=100, description="N runs/Tx", layout=ipw.Layout(width="180px"))
n_jobs_input = ipw.BoundedIntText(value=-1, min=-1, max=256, description="n_jobs (-1=all cores)", style={"description_width": "140px"}, layout=ipw.Layout(width="260px"))

block_maxiter_input = ipw.BoundedIntText(value=15, min=1, max=500, description="blk_iter", layout=ipw.Layout(width="180px"))

w_hxh_input = ipw.FloatText(value=1.0, description="w_hxh", layout=ipw.Layout(width="170px"))
w_hxhz_input = ipw.FloatText(value=2.0, description="w_hxhz", layout=ipw.Layout(width="170px"))
reg_lambda_input = ipw.FloatText(value=0.01, description="Tikhonov lambda", layout=ipw.Layout(width="200px"))

x_step_input = ipw.FloatText(value=10.0, description="dx_out", layout=ipw.Layout(width="170px"))
z_step_input = ipw.FloatText(value=5.0, description="dz_out", layout=ipw.Layout(width="170px"))
zmax_input = ipw.FloatText(value=0.0, description="zmax (0=auto)", layout=ipw.Layout(width="220px"))

load_features_btn = ipw.Button(description="Reload features", button_style="primary")
check_convergence_btn = ipw.Button(description="Check kx convergence (prior spot-check)")
run_inversion_btn = ipw.Button(description="Run all Tx inversions (calibrates per-Tx)", button_style="warning")
build_section_btn = ipw.Button(description="Build pseudo-2D section", button_style="success")
export_btn = ipw.Button(description="Export results")
quit_btn = ipw.Button(description="Quit GUI server", button_style="danger", layout=ipw.Layout(width="170px"))

qc_tx_select = ipw.Dropdown(options=[("n/a", -1)], value=-1, description="QC Tx")
qc_freq_select = ipw.Dropdown(options=[("n/a", 0)], value=0, description="QC freq")
refresh_qc_btn = ipw.Button(description="Refresh QC plots")

section_out = ipw.Output(layout=ipw.Layout(width="100%", height="640px"))
qc_out = ipw.Output(layout=ipw.Layout(width="100%", height="760px"))
convergence_out = ipw.HTML(value="Convergence check not run yet.")
status_out = ipw.Textarea(value="Ready.", description="Status", layout=ipw.Layout(width="100%", height="180px"))


def current_cfg():
    rho_min = float(rho_min_input.value)
    rho_max = float(rho_max_input.value)
    if rho_min <= 0 or rho_max <= 0 or rho_max <= rho_min:
        raise ValueError("Resistivity bounds must satisfy 0 < rho_min < rho_max")
    thk_min = float(thk_min_input.value)
    thk_max = float(thk_max_input.value)
    if thk_min <= 0 or thk_max <= thk_min:
        raise ValueError("Thickness bounds must satisfy 0 < thk_min < thk_max")

    z_start = float(z_start_input.value)
    z_end = float(z_end_input.value)
    if z_end <= z_start:
        raise ValueError("Depth bounds must satisfy z_end > z_start")
    halfspan = 0.5 * (z_end - z_start)

    return {
        "n_layers": int(n_layers_input.value),
        "eps_r": get_eps_r_used(),
        "background_rho": float(background_rho_input.value),
        "z_start": z_start,
        "z_end": z_end,
        "z_start_rel": -halfspan,
        "z_end_rel": +halfspan,
        "log10_rho_min": float(np.log10(rho_min)),
        "log10_rho_max": float(np.log10(rho_max)),
        "thk_min": thk_min,
        "thk_max": thk_max,
        "log10_thk_min": float(np.log10(thk_min)),
        "log10_thk_max": float(np.log10(thk_max)),
        "optimizer": str(optimizer_input.value),
        "calibration_mode": str(calibration_mode_input.value),
        "maxiter": int(maxiter_input.value),
        "popsize": int(popsize_input.value),
        "maxfun": int(maxfun_input.value),
        "seed": int(seed_input.value),
        "w_hxh": float(w_hxh_input.value),
        "w_hxhz": float(w_hxhz_input.value),
        "reg_lambda": float(reg_lambda_input.value),
        "x_step": float(x_step_input.value),
        "z_step": float(z_step_input.value),
        "zmax": float(zmax_input.value),
        "n_runs": int(n_runs_input.value),
        "n_jobs": int(n_jobs_input.value),
        "block_max_iter": int(block_maxiter_input.value),
    }


def on_load_features(_):
    try:
        freqs = parse_freqs(freqs_input.value)
        feats = extract_features(freqs=freqs, n_periods_extract=float(n_periods_extract_input.value))

        state["features"] = feats
        state["tx_results"] = {}
        state["section"] = None
        state["calibration"] = None

        tx_ids = sorted(feats["tx_data"].keys())
        qc_tx_select.options = [(f"Tx {i}", i) for i in tx_ids] if tx_ids else [("n/a", -1)]
        qc_tx_select.value = tx_ids[0] if tx_ids else -1
        qc_freq_select.options = [(f"{f:g} Hz", i) for i, f in enumerate(freqs)] if len(freqs) else [("n/a", 0)]
        qc_freq_select.value = 0

        push_message(f"Loaded channel-gain features from workspace/2D/forward/Data: {len(tx_ids)} Tx, {len(freqs)} freq(s).")
        push_message(f"Using eps_r_used={get_eps_r_used():.1f} from setup_metadata.json for the analytic forward.")
        if int(feats["forward_data_dim"]) == 3:
            push_message("NOTE: forward data is 3D - the 2D analytic Hx-source solver used here is not the right "
                          "reference for a 3D forward run (use the 3D empymod-based path if you need that combination).")
        return True
    except Exception as exc:
        state["features"] = None
        push_message(f"Load features failed: {exc}")
        push_message(traceback.format_exc())
        return False


def on_check_convergence(_):
    try:
        cfg = current_cfg()
        freqs = parse_freqs(freqs_input.value)
        feats = state.get("features")
        if not feats or not feats["tx_data"]:
            raise RuntimeError("Load features first.")
        tx_entry = next(iter(feats["tx_data"].values()))
        result = check_kx_convergence(
            rho_bounds=(10.0 ** cfg["log10_rho_min"], 10.0 ** cfg["log10_rho_max"]),
            thickness_bounds=(cfg["thk_min"], cfg["thk_max"]),
            n_layers=cfg["n_layers"], freqs_hz=freqs, off_x=tx_entry["off_x"],
            tx_depth_m=tx_entry["tx_z"], rx_depth_m=tx_entry["tx_z"] + float(tx_entry["off_z"][0]),
            eps_r=cfg["eps_r"], n_draws=25,
        )
        convergence_out.value = (
            f"<b>kx-grid convergence spot-check</b> ({result['n_draws']} draws over the prior): "
            f"worst relative change={result['worst_relative_change']:.2e}, "
            f"{'CONVERGED' if result['converged'] else 'NOT CONVERGED'}. {result['notes']}"
        )
        push_message("Convergence check complete - see panel above the QC section.")
    except Exception as exc:
        convergence_out.value = f"Convergence check failed: {exc}"
        push_message(f"Convergence check failed: {exc}")


def _aggregate_ensemble_runs(runs_list):
    """Aggregate a list of single-Tx inversion dicts into mean/std summaries."""
    successful = [r for r in runs_list if r.get("success", True)]
    if not successful:
        successful = runs_list

    rho_stack = np.array([r["rho"] for r in successful], dtype=float)
    thk_stack = np.array([r["thickness"] for r in successful], dtype=float)
    depth_stack = np.array([r["depth"] for r in successful], dtype=float)
    misfit_arr = np.array([r["misfit"] for r in successful], dtype=float)
    chi2_arr = np.array([r.get("chi2", np.nan) for r in successful], dtype=float)
    pred_hxh_stack = np.array([r["pred_hxh"] for r in successful], dtype=complex)
    pred_hxhz_stack = np.array([r["pred_hxhz"] for r in successful], dtype=complex)

    rho_std = np.std(rho_stack, axis=0)
    thk_std = np.std(thk_stack, axis=0)
    depth_std = np.std(depth_stack, axis=0)
    rho_unc_runs = [np.asarray(r.get("rho_unc"), dtype=float) for r in successful if r.get("rho_unc") is not None]
    thk_unc_runs = [np.asarray(r.get("thickness_unc"), dtype=float) for r in successful if r.get("thickness_unc") is not None]
    depth_unc_runs = [np.asarray(r.get("depth_unc"), dtype=float) for r in successful if r.get("depth_unc") is not None]
    if len(rho_unc_runs) == len(successful) and len(rho_unc_runs) > 0:
        rho_unc_stack = np.array(rho_unc_runs, dtype=float)
        rho_std = np.sqrt(np.maximum(rho_std ** 2 + np.mean(rho_unc_stack ** 2, axis=0), 0.0))
    if len(thk_unc_runs) == len(successful) and len(thk_unc_runs) > 0:
        thk_unc_stack = np.array(thk_unc_runs, dtype=float)
        thk_std = np.sqrt(np.maximum(thk_std ** 2 + np.mean(thk_unc_stack ** 2, axis=0), 0.0))
    if len(depth_unc_runs) == len(successful) and len(depth_unc_runs) > 0:
        depth_unc_stack = np.array(depth_unc_runs, dtype=float)
        depth_std = np.sqrt(np.maximum(depth_std ** 2 + np.mean(depth_unc_stack ** 2, axis=0), 0.0))

    best_idx = int(np.argmin(misfit_arr))

    return {
        "runs": runs_list,
        "n_successful": len(successful),
        "rho_mean": np.mean(rho_stack, axis=0),
        "rho_std": rho_std,
        "thickness_mean": np.mean(thk_stack, axis=0),
        "thickness_std": thk_std,
        "depth_mean": np.mean(depth_stack, axis=0),
        "depth_std": depth_std,
        "misfit_mean": float(np.mean(misfit_arr)),
        "misfit_std": float(np.std(misfit_arr)),
        "chi2_mean": float(np.nanmean(chi2_arr)),
        "pred_hxh_mean": np.mean(pred_hxh_stack, axis=0),
        "pred_hxh_std": np.std(pred_hxh_stack, axis=0),
        "pred_hxhz_mean": np.mean(pred_hxhz_stack, axis=0),
        "pred_hxhz_std": np.std(pred_hxhz_stack, axis=0),
        "best_run": successful[best_idx],
        "best_idx": best_idx,
    }


def _run_one_tx_seed_job(tx_id, run_idx, tx_entry, cfg_k):
    """Run a single (Tx, seed) inversion - the actual unit of work
    parallelized across the whole Tx x seed ensemble by `on_run_inversion`
    via joblib. A pure function of its own arguments (no ipywidgets/GUI
    state touched anywhere in this call chain - `invert_single_tx` and
    everything it calls only ever reads `tx_entry`/`cfg_k`), so joblib's
    default `loky` backend can ship it (by value, via cloudpickle) to a
    separate worker process without needing this notebook's code moved
    into an importable module first.
    """
    out = invert_single_tx(tx_entry, cfg_k)
    out["seed"] = cfg_k["seed"]
    out["run_idx"] = run_idx
    return tx_id, run_idx, out


def on_run_inversion(_):
    try:
        push_message("Loading phase features from workspace/2D/forward/Data...")
        ok = on_load_features(None)
        if not ok:
            raise RuntimeError("Could not load phase features automatically.")
        feats = state.get("features")
        if feats is None:
            raise RuntimeError("Load phase features first.")

        cfg = current_cfg()
        n_runs = int(cfg["n_runs"])
        base_seed = int(cfg["seed"])
        seeds = [base_seed + k for k in range(n_runs)]
        n_jobs = int(cfg.get("n_jobs", -1))

        run_dir = find_next_1d_run_dir()
        run_dir.mkdir(parents=True, exist_ok=True)
        state["active_run_dir"] = run_dir
        push_message(f"Created run folder: {run_dir.name}")

        tx_ids = sorted(feats["tx_data"].keys())
        if not tx_ids:
            raise RuntimeError("No Tx groups found in data.")

        # Calibration is cheap (closed-form fit, not an optimization) and
        # needed before a Tx's jobs can be built, so it stays a plain
        # sequential pass; only the expensive part (the actual per-(Tx,
        # seed) optimizer runs, via `_run_one_tx_seed_job`) is parallelized
        # below - every (Tx, seed) combination is fully independent of
        # every other one, so this is an embarrassingly-parallel batch.
        results = {}
        calibrations = {}
        tx_entries = {}
        jobs = []
        push_message(f"Calibrating {len(tx_ids)} Tx location(s)...")
        for tx_id in tx_ids:
            tx_entry = feats["tx_data"][tx_id]
            tx_entries[int(tx_id)] = tx_entry

            cal = run_calibration(tx_id, cfg["eps_r"])
            calibrations[int(tx_id)] = cal
            push_message(
                f"Tx {tx_id} calibration: mean rel. scatter Hx={np.mean(cal['rel_scatter_hx']) * 100:.1f}%, "
                f"Hz={np.mean(cal['rel_scatter_hz']) * 100:.1f}% ({cal['notes']})"
            )

            for k, seed_k in enumerate(seeds):
                cfg_k = dict(cfg)
                cfg_k["seed"] = seed_k
                cfg_k["calibration"] = cal
                jobs.append((tx_id, k, tx_entry, cfg_k))

        push_message(
            f"Running {len(jobs)} inversions ({len(tx_ids)} Tx x {n_runs} run(s)) in parallel "
            f"(n_jobs={n_jobs}, joblib/loky)..."
        )
        tx_runs_by_id = {int(tx_id): [] for tx_id in tx_ids}
        n_done = 0
        for tx_id, run_idx, out in Parallel(n_jobs=n_jobs, return_as="generator_unordered")(
            delayed(_run_one_tx_seed_job)(tx_id, k, tx_entry, cfg_k) for (tx_id, k, tx_entry, cfg_k) in jobs
        ):
            n_done += 1
            tx_runs_by_id[int(tx_id)].append(out)
            push_message(
                f"({n_done}/{len(jobs)}) Tx {tx_id} run {run_idx + 1}/{n_runs} (seed={out['seed']}) done."
            )

        for tx_id in tx_ids:
            tx_entry = tx_entries[int(tx_id)]
            cal = calibrations[int(tx_id)]
            tx_runs = sorted(tx_runs_by_id[int(tx_id)], key=lambda o: o["run_idx"])

            ensemble = _aggregate_ensemble_runs(tx_runs)
            best = ensemble["best_run"]
            rec = {
                "tx_id": int(tx_id),
                "tx_x": float(tx_entry["tx_x"]),
                "tx_z": float(tx_entry["tx_z"]),
                "freqs": np.asarray(tx_entry["freqs"], dtype=float),
                "obs_hx_gain": np.asarray(tx_entry["obs_hx_gain"], dtype=complex),
                "obs_hz_gain": np.asarray(tx_entry["obs_hz_gain"], dtype=complex),
                "rx_x": np.asarray(tx_entry["rx_x"], dtype=float),
                "rx_z": np.asarray(tx_entry["rx_z"], dtype=float),
                "success": best.get("success", True),
                "message": best.get("message", ""),
                "misfit": best["misfit"],
                "chi2": ensemble.get("chi2_mean", np.nan),
                "params": best["params"],
                "rho": ensemble["rho_mean"],
                "thickness": ensemble["thickness_mean"],
                "depth": ensemble["depth_mean"],
                "z_top": best["z_top"],
                "z_bottom": best["z_bottom"],
                "pred_hxh": ensemble["pred_hxh_mean"],
                "pred_hxhz": ensemble["pred_hxhz_mean"],
                "rho_std": ensemble["rho_std"],
                "thickness_std": ensemble["thickness_std"],
                "depth_std": ensemble["depth_std"],
                "misfit_mean": ensemble["misfit_mean"],
                "misfit_std": ensemble["misfit_std"],
                "pred_hxh_std": ensemble["pred_hxh_std"],
                "pred_hxhz_std": ensemble["pred_hxhz_std"],
                "n_successful": ensemble["n_successful"],
                "ensemble_runs": tx_runs,
                "calibration": cal,
            }
            results[int(tx_id)] = rec
            push_message(
                f"Tx {tx_id} done. Best misfit={best['misfit']:.4e} (chi2={rec['chi2']:.3g}), "
                f"mean={ensemble['misfit_mean']:.4e} +/- {ensemble['misfit_std']:.4e}"
            )

        state["tx_results"] = results
        state["calibration"] = calibrations

        write_1d_run_metadata(run_dir, cfg, n_runs, seeds, extra={
            "n_tx": len(tx_ids),
            "tx_ids": [int(v) for v in tx_ids],
        })

        push_message(f"All Tx ensemble inversions completed ({n_runs} run(s)/Tx). Results in {run_dir.name}.")

        # Auto-save NPZ/JSON for this run (disable SEGY/RSS to keep it an internal step).
        # This makes the ensemble mean/std available in `06_1d_empymod_results.ipynb`.
        state["skip_segy_export"] = True
        try:
            on_export(None)
        except Exception as exc:
            push_message(f"Auto-save (NPZ/JSON) failed: {exc}")
            push_message(traceback.format_exc())
        finally:
            state["skip_segy_export"] = False
    except Exception as exc:
        push_message(f"Run inversion failed: {exc}")
        push_message(traceback.format_exc())


def _build_ensemble_sections(tx_results, x_grid, z_grid, background_rho):
    """Build per-run pseudo-2D sections and aggregate to mean/std."""
    first = next(iter(tx_results.values()))
    ensemble_runs = first.get("ensemble_runs")
    n_runs = len(ensemble_runs) if ensemble_runs else 1

    if n_runs <= 1:
        sec = interpolate_tx_profiles(tx_results, x_grid, z_grid, background_rho)
        return sec, np.zeros_like(sec)

    sections = []
    for k in range(n_runs):
        run_k_results = {}
        for tx_id, rec in tx_results.items():
            runs_list = rec.get("ensemble_runs", [])
            if k < len(runs_list):
                single = dict(runs_list[k])
            else:
                single = dict(rec)
            single["tx_x"] = rec["tx_x"]
            single["tx_z"] = rec["tx_z"]
            single["rx_x"] = rec.get("rx_x", np.array([]))
            run_k_results[tx_id] = single
        sec_k = interpolate_tx_profiles(run_k_results, x_grid, z_grid, background_rho)
        sections.append(sec_k)

    stack = np.array(sections, dtype=float)
    return np.mean(stack, axis=0), np.std(stack, axis=0)


def on_build_section(_):
    try:
        tx_results = state.get("tx_results") or {}
        if not tx_results:
            raise RuntimeError("Run Tx inversions first.")

        cfg = current_cfg()
        x_default, z_default = get_default_model_axes()
        if x_default is not None and z_default is not None:
            x_model = np.asarray(x_default, dtype=float)
            z_model = np.asarray(z_default, dtype=float)
        else:
            x_model = try_default_x_grid(FWD_2D_DIR)
            if x_model is None:
                xs = np.array([float(v["tx_x"]) for v in tx_results.values()], dtype=float)
                xs.sort()
                dx = max(1.0, float(cfg["x_step"]))
                x_model = np.arange(xs.min(), xs.max() + dx, dx)

            dz = max(0.1, float(cfg["z_step"]))
            zmax = None if cfg["zmax"] <= 0 else float(cfg["zmax"])
            z_model = build_z_grid_from_results(tx_results, dz=dz, zmax=zmax)

        rho_mean, rho_std = _build_ensemble_sections(
            tx_results,
            x_grid=np.asarray(x_model, dtype=float),
            z_grid=np.asarray(z_model, dtype=float),
            background_rho=float(cfg["background_rho"]),
        )

        tx_x, tx_z, rx_x, rx_z = _extract_positions()

        state["section"] = {
            "x": np.asarray(x_model, dtype=float),
            "z": np.asarray(z_model, dtype=float),
            "rho": rho_mean,
            "rho_mean": rho_mean,
            "rho_std": rho_std,
            "background_rho": float(cfg["background_rho"]),
            "tx_x": tx_x,
            "tx_z": tx_z,
            "rx_x": rx_x,
            "rx_z": rx_z,
        }

        rho_true = None
        x_true = None
        z_true = None
        if SG_TRUE_PATH.exists():
            try:
                x_true, z_true, g_true = _read_rss_model(SG_TRUE_PATH)
                rho_true = conductivity_to_resistivity(g_true)
            except Exception as exc:
                push_message(f"True-model load warning: {exc}")

        has_std = np.any(rho_std > 0)

        if rho_true is not None:
            finite_vals = [rho_mean, np.asarray(rho_true, dtype=float)]
            finite = np.concatenate([v[np.isfinite(v)] for v in finite_vals if v.size])
            zmin_c = float(np.nanmin(finite)) if finite.size else None
            zmax_c = float(np.nanmax(finite)) if finite.size else None

            ncols = 3 if has_std else 2
            titles = ["Pseudo-2D mean", "True model (workspace/2D/forward/sg.rss)"]
            if has_std:
                titles.insert(1, "Pseudo-2D std")

            fig = make_subplots(
                rows=1,
                cols=ncols,
                subplot_titles=titles,
                horizontal_spacing=0.06,
                shared_xaxes="all",
                shared_yaxes="all",
            )

            fig.add_trace(
                go.Heatmap(
                    x=x_model, y=z_model, z=rho_mean,
                    colorscale="Viridis", zmin=zmin_c, zmax=zmax_c, showscale=False,
                ),
                row=1, col=1,
            )

            if has_std:
                fig.add_trace(
                    go.Heatmap(
                        x=x_model, y=z_model, z=rho_std,
                        colorscale="Hot", showscale=True,
                        colorbar=dict(title="Std", x=0.60 if ncols == 3 else 1.02, len=0.92, thickness=12),
                    ),
                    row=1, col=2,
                )

            true_col = ncols
            fig.add_trace(
                go.Heatmap(
                    x=x_true, y=z_true, z=rho_true,
                    colorscale="Viridis", zmin=zmin_c, zmax=zmax_c,
                    showscale=True,
                    colorbar=dict(title="Ohm.m", x=1.02, len=0.92, thickness=12),
                ),
                row=1, col=true_col,
            )

            for col in range(1, ncols + 1):
                if tx_x.size:
                    fig.add_trace(
                        go.Scatter(
                            x=tx_x, y=tx_z, mode="markers",
                            marker=dict(symbol="triangle-up", size=7, color="red"),
                            name="TX", showlegend=(col == 1),
                        ),
                        row=1, col=col,
                    )
                if rx_x.size:
                    fig.add_trace(
                        go.Scatter(
                            x=rx_x, y=rx_z, mode="markers",
                            marker=dict(symbol="circle", size=6, color="cyan"),
                            name="RX", showlegend=(col == 1),
                        ),
                        row=1, col=col,
                    )

            for col in range(1, ncols + 1):
                fig.update_xaxes(title_text="x (m)", row=1, col=col)
                fig.update_yaxes(title_text="z (m)", autorange="reversed", row=1, col=col)
            fig.update_xaxes(matches="x")
            fig.update_yaxes(matches="y")
            fig.update_layout(
                title="Pseudo-2D section and true model",
                height=620,
                margin=dict(t=70, b=50, l=60, r=100),
                legend=dict(orientation="h", y=-0.12),
            )
        else:
            ncols = 2 if has_std else 1
            titles = ["Pseudo-2D mean"]
            if has_std:
                titles.append("Pseudo-2D std")

            if ncols == 1:
                fig = go.Figure()
                fig.add_trace(
                    go.Heatmap(
                        x=x_model, y=z_model, z=rho_mean,
                        colorscale="Viridis", colorbar=dict(title="Ohm.m"),
                    )
                )
                if tx_x.size:
                    fig.add_trace(go.Scatter(
                        x=tx_x, y=tx_z, mode="markers",
                        marker=dict(symbol="triangle-up", size=7, color="red"), name="TX",
                    ))
                if rx_x.size:
                    fig.add_trace(go.Scatter(
                        x=rx_x, y=rx_z, mode="markers",
                        marker=dict(symbol="circle", size=6, color="cyan"), name="RX",
                    ))
                fig.update_layout(
                    title="Pseudo-2D section from per-Tx 1D phase-only inversion",
                    xaxis_title="x (m)", yaxis_title="z (m)",
                    yaxis_autorange="reversed", height=560,
                    margin=dict(t=60, b=50, l=60, r=20),
                    legend=dict(orientation="h", y=-0.15),
                )
            else:
                fig = make_subplots(
                    rows=1, cols=2, subplot_titles=titles,
                    horizontal_spacing=0.08, shared_xaxes="all", shared_yaxes="all",
                )
                fig.add_trace(
                    go.Heatmap(
                        x=x_model, y=z_model, z=rho_mean,
                        colorscale="Viridis", showscale=True,
                        colorbar=dict(title="Ohm.m", x=0.45, len=0.92),
                    ),
                    row=1, col=1,
                )
                fig.add_trace(
                    go.Heatmap(
                        x=x_model, y=z_model, z=rho_std,
                        colorscale="Hot", showscale=True,
                        colorbar=dict(title="Std", x=1.02, len=0.92),
                    ),
                    row=1, col=2,
                )
                for col in (1, 2):
                    if tx_x.size:
                        fig.add_trace(go.Scatter(
                            x=tx_x, y=tx_z, mode="markers",
                            marker=dict(symbol="triangle-up", size=7, color="red"),
                            name="TX", showlegend=(col == 1),
                        ), row=1, col=col)
                    if rx_x.size:
                        fig.add_trace(go.Scatter(
                            x=rx_x, y=rx_z, mode="markers",
                            marker=dict(symbol="circle", size=6, color="cyan"),
                            name="RX", showlegend=(col == 1),
                        ), row=1, col=col)
                for col in (1, 2):
                    fig.update_xaxes(title_text="x (m)", row=1, col=col)
                    fig.update_yaxes(title_text="z (m)", autorange="reversed", row=1, col=col)
                fig.update_xaxes(matches="x")
                fig.update_yaxes(matches="y")
                fig.update_layout(
                    title="Pseudo-2D section (mean and std)",
                    height=620, margin=dict(t=70, b=50, l=60, r=100),
                    legend=dict(orientation="h", y=-0.12),
                )

        with section_out:
            section_out.clear_output(wait=True)
            display(fig)

        push_message("Pseudo-2D section built and plotted.")

        # Auto-save NPZ/JSON after building the pseudo-2D section.
        state["skip_segy_export"] = True
        try:
            on_export(None)
        except Exception as exc:
            push_message(f"Auto-save (NPZ/JSON) after section failed: {exc}")
            push_message(traceback.format_exc())
        finally:
            state["skip_segy_export"] = False
    except Exception as exc:
        push_message(f"Build section failed: {exc}")
        push_message(traceback.format_exc())


def on_refresh_qc(_):
    push_message("Refreshing QC...")
    with qc_out:
        qc_out.clear_output(wait=False)
        try:
            tx_id = int(qc_tx_select.value)
            if tx_id < 0:
                display(ipw.HTML("No Tx selected."))
                return
            res = (state.get("tx_results") or {}).get(tx_id)
            if res is None:
                display(ipw.HTML("Run inversion first to populate QC for this Tx."))
                return

            freqs = np.asarray(res["freqs"], dtype=float)
            obs_hx = np.asarray(res["obs_hx_gain"], dtype=complex)
            obs_hz = np.asarray(res["obs_hz_gain"], dtype=complex)
            pred_hx = np.asarray(res["pred_hxh"], dtype=complex)
            pred_hz = np.asarray(res["pred_hxhz"], dtype=complex)

            if freqs.size == 0 or obs_hx.ndim != 2 or obs_hz.ndim != 2:
                display(ipw.HTML("No valid QC data available."))
                return

            fidx = int(qc_freq_select.value) if qc_freq_select.value is not None else 0
            fidx = max(0, min(fidx, freqs.size - 1))

            cal = res.get("calibration") or {}
            cal_mode = str(getattr(calibration_mode_input, "value", "fixed"))
            pred_hx_cal, pred_hz_cal = _apply_calibration(pred_hx, obs_hx, pred_hz, obs_hz, cal_mode, cal.get("C"))

            true_hx, true_hz = forward_analytic_true_model_for_tx(
                (state.get("features") or {}).get("tx_data", {}).get(tx_id, {}), get_eps_r_used(),
            )

            x_idx = np.arange(obs_hx.shape[1])
            fig = make_subplots(
                rows=2, cols=2,
                subplot_titles=[
                    f"Hx amplitude (f={freqs[fidx]:g} Hz)", f"Hz amplitude (f={freqs[fidx]:g} Hz)",
                    "Hx phase (deg)", "Hz phase (deg)",
                ],
                horizontal_spacing=0.10, vertical_spacing=0.12,
            )
            for col, (obs, pred, pred_c, true) in enumerate(
                [(obs_hx, pred_hx, pred_hx_cal, true_hx), (obs_hz, pred_hz, pred_hz_cal, true_hz)], start=1
            ):
                sl = col == 1
                fig.add_trace(go.Scatter(x=x_idx, y=np.abs(obs[fidx]), mode="lines+markers", name="Obs", showlegend=sl), row=1, col=col)
                fig.add_trace(go.Scatter(x=x_idx, y=np.abs(pred_c[fidx]), mode="lines+markers", name="Pred (calibrated)", showlegend=sl), row=1, col=col)
                fig.add_trace(go.Scatter(x=x_idx, y=np.abs(true[fidx]), mode="lines", name="True (uncalibrated)", showlegend=sl, line=dict(dash="dot")), row=1, col=col)
                fig.add_trace(go.Scatter(x=x_idx, y=np.angle(obs[fidx], deg=True), mode="lines+markers", name="Obs", showlegend=False), row=2, col=col)
                fig.add_trace(go.Scatter(x=x_idx, y=np.angle(pred_c[fidx], deg=True), mode="lines+markers", name="Pred", showlegend=False), row=2, col=col)
                fig.add_trace(go.Scatter(x=x_idx, y=np.angle(true[fidx], deg=True), mode="lines", name="True", showlegend=False, line=dict(dash="dot")), row=2, col=col)

            for c in (1, 2):
                fig.update_xaxes(title_text="Local rx index", row=2, col=c)
                fig.update_yaxes(title_text="Amplitude", row=1, col=c)
                fig.update_yaxes(title_text="Phase (deg)", range=[-180.0, 180.0], row=2, col=c)
            fig.update_layout(
                title=f"Tx {tx_id} channel-gain QC (calibration={cal_mode})",
                height=760, margin=dict(t=60, b=45, l=60, r=20), legend=dict(orientation="h", y=-0.12),
            )
            display(fig)

            tx_ids = sorted((state.get("tx_results") or {}).keys())
            if tx_ids:
                mis = [float(state["tx_results"][k]["misfit"]) for k in tx_ids]
                chi2 = [float(state["tx_results"][k].get("chi2", np.nan)) for k in tx_ids]
                tx_x = [float(state["tx_results"][k]["tx_x"]) for k in tx_ids]
                fig2 = make_subplots(rows=1, cols=2, subplot_titles=["Misfit vs Tx position", "chi2 vs Tx position"])
                fig2.add_trace(go.Scatter(x=tx_x, y=mis, mode="lines+markers", name="Misfit", showlegend=False), row=1, col=1)
                fig2.add_trace(go.Scatter(x=tx_x, y=chi2, mode="lines+markers", name="chi2", showlegend=False), row=1, col=2)
                fig2.add_hline(y=1.0, line=dict(dash="dash", color="gray"), row=1, col=2)
                fig2.update_xaxes(title_text="Tx x (m)")
                fig2.update_layout(height=260, margin=dict(t=40, b=40, l=60, r=20))
                display(fig2)

            display(ipw.HTML(
                f"<b>Tx {tx_id}</b> best misfit: {res['misfit']:.4e}, chi2={res.get('chi2', float('nan')):.3g} "
                "(chi2~1 means the fit matches the calibrated noise floor - not overfitting past it)."
            ))
        except Exception as exc:
            display(ipw.HTML(f"QC plot failed: {exc}"))


def on_export(_):
    try:
        run_dir = state.get("active_run_dir")
        if run_dir is None:
            run_dir = find_next_1d_run_dir()
            run_dir.mkdir(parents=True, exist_ok=True)
            state["active_run_dir"] = run_dir
        out_dir = Path(run_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        skip_segy_export = bool(state.get("skip_segy_export", False))

        tx_results = state.get("tx_results") or {}
        section = state.get("section")
        if not tx_results:
            raise RuntimeError("No inversion results to export.")

        # Export/QC should not depend on inversion-bound validation.
        # If current_cfg() raises (e.g. while editing bounds), fall back to widget values.
        try:
            cfg = current_cfg()
        except Exception:
            cfg = {
                "n_layers": int(n_layers_input.value),
                "eps_r": get_eps_r_used(),
                "background_rho": float(background_rho_input.value),
                "z_start": float(z_start_input.value),
                "z_end": float(z_end_input.value),
                "optimizer": str(optimizer_input.value),
                "calibration_mode": str(calibration_mode_input.value),
                "seed": int(seed_input.value),
                "w_hxh": float(w_hxh_input.value),
                "w_hxhz": float(w_hxhz_input.value),
                "reg_lambda": float(reg_lambda_input.value),
                "x_step": float(x_step_input.value),
                "z_step": float(z_step_input.value),
                "zmax": float(zmax_input.value),
                "n_runs": int(n_runs_input.value),
            }
        n_runs = int(cfg.get("n_runs", int(n_runs_input.value)))
        tx_ids = sorted(tx_results.keys())
        max_layers = max(int(tx_results[k]["rho"].size) for k in tx_ids)
        max_thk = max(int(tx_results[k]["thickness"].size) for k in tx_ids)

        tx_x = np.array([tx_results[k]["tx_x"] for k in tx_ids], dtype=float)
        tx_z = np.array([tx_results[k]["tx_z"] for k in tx_ids], dtype=float)
        misfit = np.array([tx_results[k]["misfit"] for k in tx_ids], dtype=float)

        rho_arr = np.full((len(tx_ids), max_layers), np.nan, dtype=float)
        thk_arr = np.full((len(tx_ids), max_thk), np.nan, dtype=float)
        rho_std_arr = np.full((len(tx_ids), max_layers), np.nan, dtype=float)
        thk_std_arr = np.full((len(tx_ids), max_thk), np.nan, dtype=float)
        misfit_mean_arr = np.zeros(len(tx_ids), dtype=float)
        misfit_std_arr = np.zeros(len(tx_ids), dtype=float)
        for i, k in enumerate(tx_ids):
            rec = tx_results[k]
            r = np.asarray(rec["rho"], dtype=float)
            t = np.asarray(rec["thickness"], dtype=float)
            rho_arr[i, :r.size] = r
            thk_arr[i, :t.size] = t
            if "rho_std" in rec:
                rs = np.asarray(rec["rho_std"], dtype=float)
                rho_std_arr[i, :rs.size] = rs
            if "thickness_std" in rec:
                ts = np.asarray(rec["thickness_std"], dtype=float)
                thk_std_arr[i, :ts.size] = ts
            misfit_mean_arr[i] = float(rec.get("misfit_mean", rec["misfit"]))
            misfit_std_arr[i] = float(rec.get("misfit_std", 0.0))

        npz_data = {
            "tx_ids": np.asarray(tx_ids, dtype=int),
            "tx_x": tx_x,
            "tx_z": tx_z,
            "misfit": misfit,
            "misfit_mean": misfit_mean_arr,
            "misfit_std": misfit_std_arr,
            "rho_layers": rho_arr,
            "rho_layers_std": rho_std_arr,
            "thickness_layers": thk_arr,
            "thickness_layers_std": thk_std_arr,
            "n_runs": np.array(n_runs, dtype=int),
        }

        if section is not None:
            npz_data["section_x"] = np.asarray(section["x"], dtype=float)
            npz_data["section_z"] = np.asarray(section["z"], dtype=float)
            npz_data["section_rho"] = np.asarray(section.get("rho_mean", section["rho"]), dtype=float)
            if "rho_std" in section:
                npz_data["section_rho_std"] = np.asarray(section["rho_std"], dtype=float)

        chi2_arr = np.array([tx_results[k].get("chi2", np.nan) for k in tx_ids], dtype=float)
        npz_data["chi2"] = chi2_arr

        # Save per-Tx complex gains (obs/pred, mean+std) and calibration for the results notebook.
        for k_idx, k in enumerate(tx_ids):
            rec = tx_results[k]
            if "pred_hxh" in rec:
                npz_data[f"pred_hxh_mean_tx{k}"] = np.asarray(rec["pred_hxh"], dtype=complex)
            if "pred_hxhz" in rec:
                npz_data[f"pred_hxhz_mean_tx{k}"] = np.asarray(rec["pred_hxhz"], dtype=complex)
            if "pred_hxh_std" in rec:
                npz_data[f"pred_hxh_std_tx{k}"] = np.asarray(rec["pred_hxh_std"], dtype=float)
            if "pred_hxhz_std" in rec:
                npz_data[f"pred_hxhz_std_tx{k}"] = np.asarray(rec["pred_hxhz_std"], dtype=float)
            if "obs_hx_gain" in rec:
                npz_data[f"obs_hx_gain_tx{k}"] = np.asarray(rec["obs_hx_gain"], dtype=complex)
            if "obs_hz_gain" in rec:
                npz_data[f"obs_hz_gain_tx{k}"] = np.asarray(rec["obs_hz_gain"], dtype=complex)
            if "freqs" in rec:
                npz_data[f"freqs_tx{k}"] = np.asarray(rec["freqs"], dtype=float)
            if "rx_x" in rec:
                npz_data[f"rx_x_tx{k}"] = np.asarray(rec["rx_x"], dtype=float)
            cal = rec.get("calibration")
            if cal is not None:
                npz_data[f"sigma_hx_tx{k}"] = np.asarray(cal["sigma_hx"], dtype=float)
                npz_data[f"sigma_hz_tx{k}"] = np.asarray(cal["sigma_hz"], dtype=float)
                if cal.get("C") is not None:
                    npz_data[f"C_tx{k}"] = np.asarray(cal["C"], dtype=complex)

        npz_path = out_dir / "analytic_1d_inversion_results.npz"
        np.savez(npz_path, **npz_data)

        summary = {
            "run_dir": str(out_dir),
            "n_tx": int(len(tx_ids)),
            "tx_ids": [int(v) for v in tx_ids],
            "n_runs_per_tx": n_runs,
            "misfit": [float(v) for v in misfit],
            "misfit_mean": [float(v) for v in misfit_mean_arr],
            "misfit_std": [float(v) for v in misfit_std_arr],
            "chi2": [float(v) for v in chi2_arr],
            "optimizer": str(optimizer_input.value),
            "calibration_mode": str(cfg.get("calibration_mode", "fixed")),
            "n_layers": int(n_layers_input.value),
            "eps_r_used": float(cfg.get("eps_r", get_eps_r_used())),
            "reg_lambda": float(cfg.get("reg_lambda", reg_lambda_input.value)),
            "freqs_hz": [float(v) for v in parse_freqs(freqs_input.value)],
            "seed_base": int(seed_input.value),
        }
        json_path = out_dir / "analytic_1d_inversion_summary.json"
        json_path.write_text(json.dumps(summary, indent=2))

        if section is not None and not skip_segy_export:
            rho_for_export = np.asarray(section.get("rho_mean", section["rho"]), dtype=float)
            rho_std_for_export = np.asarray(section.get("rho_std"), dtype=float) if "rho_std" in section else None
            template = FWD_2D_DIR / "sg0.rss"
            if not template.exists() and SG_TRUE_PATH.exists():
                template = SG_TRUE_PATH
            if template.exists():
                try:
                    rss_path = out_dir / "analytic_1d_pseudo2d_sigma.rss"
                    write_rho_section_to_rss(
                        template,
                        rss_path,
                        rho_std_for_export if rho_std_for_export is not None else rho_for_export,
                    )
                    if rho_std_for_export is not None:
                        push_message(f"RSS exported (std section): {rss_path}")
                    else:
                        push_message(f"RSS exported (mean section): {rss_path}")
                except Exception as exc:
                    push_message(f"RSS export skipped: {exc}")

            template_segy, template_src = _resolve_segy_template_path()
            if template_segy is not None:
                try:
                    segy_path = out_dir / "analytic_1d_pseudo2d.segy"
                    info = write_resistivity_to_segy_from_template(
                        template_segy_path=template_segy,
                        output_segy_path=segy_path,
                        resistivity_grid=rho_for_export,
                        x_model=np.asarray(section["x"], dtype=float),
                        z_model=np.asarray(section["z"], dtype=float),
                        method="linear",
                    )
                    push_message(f"SEGY export SUCCESS ({template_src}): {segy_path}")
                    if info.get("interpolated"):
                        push_message("SEGY export used interpolation to match template geometry.")
                    else:
                        push_message("SEGY export grid already matched template geometry.")

                    if rho_std_for_export is not None:
                        try:
                            segy_std_path = out_dir / "analytic_1d_pseudo2d_std.segy"
                            write_resistivity_to_segy_from_template(
                                template_segy_path=template_segy,
                                output_segy_path=segy_std_path,
                                resistivity_grid=rho_std_for_export,
                                x_model=np.asarray(section["x"], dtype=float),
                                z_model=np.asarray(section["z"], dtype=float),
                                method="linear",
                            )
                            push_message(f"SEGY std export SUCCESS ({template_src}): {segy_std_path}")
                        except Exception as exc2:
                            push_message(f"SEGY std export failed: {exc2}")
                except Exception as exc:
                    push_message(f"SEGY export failed: {exc}")
            else:
                push_message("SEGY export skipped: no SEG-Y template found.")

        push_message(f"Export complete: {npz_path}")
        push_message(f"Summary JSON: {json_path}")
        push_message(f"Output folder: {out_dir}")
    except Exception as exc:
        push_message(f"Export failed: {exc}")
        push_message(traceback.format_exc())


def on_quit_gui(_):
    push_message("Shutting down 1D inversion GUI server...")

    try:
        if VOILA_PID_FILE.exists():
            pid_text = VOILA_PID_FILE.read_text().strip()
            if pid_text:
                pid = int(pid_text)
                if pid != os.getpid():
                    try:
                        os.kill(pid, signal.SIGTERM)
                        push_message(f"Sent SIGTERM to Voila server PID {pid}.")
                    except Exception as exc:
                        push_message(f"Could not signal Voila PID {pid}: {exc}")
    except Exception as exc:
        push_message(f"Quit warning: {exc}")

    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip and getattr(ip, "kernel", None):
            ip.kernel.do_shutdown(restart=False)
    except Exception as exc:
        push_message(f"Kernel shutdown warning: {exc}")

    try:
        os.kill(os.getpid(), signal.SIGTERM)
    except Exception:
        pass


def _rebind_on_click(btn, handler):
    """Ensure the visible widget has exactly one handler bound."""
    try:
        # ipywidgets stores callbacks here
        callbacks = list(getattr(btn, "_click_handlers").callbacks)
        for cb in callbacks:
            btn.on_click(cb, remove=True)
    except Exception:
        pass
    btn.on_click(handler)


_rebind_on_click(load_features_btn, on_load_features)
_rebind_on_click(check_convergence_btn, on_check_convergence)
_rebind_on_click(run_inversion_btn, on_run_inversion)
_rebind_on_click(build_section_btn, on_build_section)
_rebind_on_click(refresh_qc_btn, on_refresh_qc)
_rebind_on_click(export_btn, on_export)
_rebind_on_click(quit_btn, on_quit_gui)

app = ipw.VBox([
    ipw.HTML("<h2>05_1d_inversion (analytic magnetic line-source solver)</h2>"),
    ipw.HBox([quit_btn], layout=ipw.Layout(justify_content="flex-end")),
    ipw.HTML(
        "<p>1D layered inversion per Tx against the validated analytic Hx-source line-source solver "
        "(rockem-suite's <code>magnetic_line_source_fields_layered</code>) - no empymod, no phase/amplitude "
        "correction constants. Each Tx is calibrated against the exact model that produced its own FDTD data "
        "before inversion, and a Tikhonov penalty discourages overfitting past the calibrated noise floor.</p>"
    ),
    ipw.HTML("<h3>1) Load channel-gain features</h3>"),
    ipw.HBox([freqs_input, n_periods_extract_input, load_features_btn, check_convergence_btn]),
    convergence_out,
    ipw.HTML("<h3>2) 1D model parameterization and inversion settings</h3>"),
    ipw.HBox([n_layers_input, z_start_input, z_end_input, background_rho_input]),
    ipw.HBox([rho_min_input, rho_max_input, thk_min_input, thk_max_input]),
    ipw.HBox([optimizer_input, calibration_mode_input, maxiter_input, popsize_input, maxfun_input, seed_input, n_runs_input, n_jobs_input]),
    ipw.HBox([block_maxiter_input, w_hxh_input, w_hxhz_input, reg_lambda_input]),
    ipw.HBox([run_inversion_btn]),
    ipw.HTML("<h3>3) Build pseudo-2D section</h3>"),
    ipw.HBox([build_section_btn]),
    section_out,
    ipw.HTML("<h3>4) QC and export</h3>"),
    ipw.HBox([qc_tx_select, qc_freq_select, refresh_qc_btn, export_btn]),
    qc_out,
    status_out,
])

display(app)
push_message("Ready. Observed channel gains are always loaded from workspace/2D/forward/Data (2D inversion input fallback only if needed).")
push_message("Outputs are written to workspace/1D/inversion/OneDRunN/ (not inside workspace/2D/forward).")
push_message("Each Tx is calibrated (fixed constant or per-frequency source factor) against the exact model "
              "beneath it before inversion - see 'Run all Tx inversions' log for per-Tx calibration scatter.")
on_load_features(None)
